In [1]:
import gspread
import pandas as pd
from google.oauth2.service_account import Credentials

## Pipeline A

In [2]:
SERVICE_ACCOUNT_FILE = '../../key/credentials.json'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets',
          'https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
client = gspread.authorize(creds)

sheet = client.open("2025 Analyze (Like/Dislike)").worksheet("Raw Data [Statistic form] 2024")
data = sheet.get_all_values()

df = pd.DataFrame(data[1:], columns=data[0])
df.to_csv('../../raw_data/like_dislike.csv', index=False)

In [3]:
df_clean = df.copy()

# Hapus spasi berlebih
df_clean.columns = df_clean.columns.str.strip()
df_clean.columns = df_clean.columns.str.replace(r'\s+', ' ', regex=True)

# Change to datetime
datetime_col = [
    'Date'
]
for col in datetime_col:
    if col in df_clean.columns:
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')

# Replacing the empty data with null
df_clean['business'].replace('', 'no data', inplace=True)

# Check is there any NA
if df_clean['business'].isna().any():
    df_clean['business'] = df_clean['business'].fillna('no data')

# Convert to categorycal data type
categorical_cols = [
    'business', 'Manual Check [business]', 'Team/Category', 'Week'
]
for col in categorical_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('category')

# Convert to numerical data type
numerical_cols = [
    'solved_num','unsolved_num'
]
for col in numerical_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

C:\Users\ozy.prazuganda\AppData\Local\Temp\ipykernel_17736\1173681464.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['business'].replace('', 'no data', inplace=True)


In [4]:
df_clean

,Date,business,code_type,Code,solved_num,unsolved_num,Manual Check [business],Background detail,Team/Category,Week,
0,2024-01-01,no data,高频FAQ,289-292-446-173,18,18,AFI,Failed to apply for a loan,Loan,,
1,2024-01-01,AFI,高频FAQ,289-292-446-173,10,7,AFI,Failed to apply for a loan,Loan,,
2,2024-01-01,no data,模型匹配,289-377-534-617,1,1,AFI,Loan application tenor,Loan,,
3,2024-01-01,AFI,模型匹配,738-755-774-837,1,1,AFI,Submission to unblock the account,Account security,,
4,2024-01-01,AFI,模型匹配,289-409-566-712,1,3,AFI,Want a loan application,Loan,,
...,...,...,...,...,...,...,...,...,...,...,...
226908,2026-06-18,4,模型匹配,289-1852-1853-1232,2,0,ASI,Loan fine relief/discount,Loan,W3 Jun 2026,
226909,2026-06-18,no data,模型匹配,0,43,42,#N/A,#N/A,#N/A,W3 Jun 2026,
226910,2026-06-18,no data,模型匹配,289-398-555-656,1,0,AFI,Paylater submission failed,Submission of Limit/Credit Points,W3 Jun 2026,
226911,2026-06-18,AFI,模型匹配,1334-1335-1113,1,0,AFI/ASI,Chit-chat,No Category,W3 Jun 2026,


In [5]:
df_clean.to_csv('../../dataset_kula/kula_like_dislike.csv')